# Lab 4 · An honest backtest

**Week 4 · Backtesting and execution** · ≈ 90 minutes

You will test 18 variants of a long-only momentum strategy on the 25 Asia-Pacific portfolios, choose
one using **only** 1991–2010, pay realistic Hong Kong-style trading costs, and then look at what happens
from 2011 onwards. Finally you will compute the deflated Sharpe ratio for your choice.

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

**How this lab works.** Run the setup cell, then work top to bottom. Each exercise has a `# TODO` cell followed by an automatic check (✓ / ✗ with a hint). When every check passes, the last cell prints a **completion code**: paste it into the course page to complete the lab.

In [ ]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")

def _reference_dsr(trials, best, train):
    from scipy.stats import kurtosis, norm, skew
    srs = np.array([t[train].mean() / t[train].std() for t in trials.values()])
    x = trials[best][train]
    s, n = x.mean() / x.std(), len(srs)
    sr0 = srs.std(ddof=1) * ((1 - 0.5772156649) * norm.ppf(1 - 1 / n) + 0.5772156649 * norm.ppf(1 - 1 / (n * np.e)))
    return norm.cdf((s - sr0) * np.sqrt(len(x) - 1) / np.sqrt(1 - skew(x) * s + (kurtosis(x, fisher=False) - 1) / 4 * s ** 2))


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from itertools import product
from scipy.stats import kurtosis, norm, skew

returns = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
rf = load_french("Asia_Pacific_ex_Japan_3_Factors")["RF"]
COST_PER_SIDE = 0.0025   # 25 bps: 10 bps stamp duty + half-spread + commission + impact
TRAIN, TEST = slice("1991-01", "2010-12"), slice("2011-01", None)

def momentum_strategy(returns, lookback, top_k, hold):
    """Long-only: equal-weight the top_k portfolios by past return, rebalanced every `hold` months."""
    score = (1 + returns).rolling(lookback - 1).apply(np.prod, raw=True).shift(1) - 1
    rows, weights, w_prev = [], [], np.zeros(returns.shape[1])
    for i in range(lookback + 1, len(returns)):
        if (i - lookback - 1) % hold == 0:
            w = np.zeros(returns.shape[1])
            w[returns.columns.get_indexer(score.iloc[i - 1].nlargest(top_k).index)] = 1 / top_k
        else:                                  # let weights drift between rebalances
            w = w_prev * (1 + returns.iloc[i - 1].values)
            w = w / w.sum()
        turnover = np.abs(w - w_prev).sum() / 2 if i > lookback + 1 else 1.0
        rows.append((returns.index[i], w @ returns.iloc[i].values, turnover))
        weights.append(w)
        w_prev = w
    frame = pd.DataFrame(rows, columns=["month", "ret", "turnover"]).set_index("month")
    return frame, pd.DataFrame(weights, index=frame.index, columns=returns.columns)

## Exercise 1 · Run every trial and log it

Grid: lookback ∈ {6, 9, 12}, top_k ∈ {3, 5, 8}, hold ∈ {1, 3} (18 trials). For each, compute the
monthly **net excess return** = return − rf − turnover × 2 × cost (buying and selling).

In [ ]:
trials = None  # TODO: dict {(lookback, top_k, hold): net excess return Series}

In [ ]:
check("All 18 trials logged", isinstance(trials, dict) and len(trials) == 18, "three lookbacks × three top_k × two holds")

## Exercise 2 · Select in-sample, evaluate out-of-sample

Pick the trial with the highest **monthly** Sharpe ratio over `TRAIN` only, then report its annualised
Sharpe in `TRAIN` and in `TEST`.

In [ ]:
best = None        # TODO: the winning (lookback, top_k, hold)
train_sharpe = None  # TODO: annualised
test_sharpe = None   # TODO: annualised

In [ ]:
reference_best = max(trials, key=lambda k: trials[k][TRAIN].mean() / trials[k][TRAIN].std())
check("Selection used the training period only", best == reference_best,
      "rank trials on trials[key][TRAIN], never on the full sample")
check("Out-of-sample Sharpe is right",
      np.isclose(test_sharpe, trials[best][TEST].mean() / trials[best][TEST].std() * np.sqrt(12)),
      "annualise a monthly Sharpe with √12")

## Exercise 3 · The deflated Sharpe ratio

Bailey & López de Prado (2014). With $\hat{SR}$ the best *monthly* training Sharpe, $T$ its number of
months, γ₃ skewness and γ₄ (non-excess) kurtosis of its returns, and $SR_0$ the expected maximum Sharpe of
N skill-less trials:

$$SR_0 = \sqrt{V[\hat{SR}_n]}\,\Big[(1-\gamma)\Phi^{-1}(1-\tfrac1N) + \gamma\,\Phi^{-1}(1-\tfrac{1}{Ne})\Big],\qquad
DSR = \Phi\!\left(\frac{(\hat{SR}-SR_0)\sqrt{T-1}}{\sqrt{1-\gamma_3\hat{SR}+\frac{\gamma_4-1}{4}\hat{SR}^2}}\right)$$

In [ ]:
dsr = None  # TODO: probability that the selected strategy's true Sharpe is above SR0

In [ ]:
check("Deflated Sharpe ratio is a probability", dsr is not None and 0 <= dsr <= 1, "wrap the statistic in norm.cdf")
check("Deflated Sharpe ratio is correctly computed", dsr is not None and np.isclose(dsr, _reference_dsr(trials, best, TRAIN), atol=1e-6),
      "use monthly Sharpe ratios, ddof=1 for the variance, and non-excess kurtosis")
completion_code("lab4")

## Reflect

* Your choice passes the deflated Sharpe test in-sample, yet its Sharpe falls sharply after 2010. Name three explanations.
* What is the annual cost drag at your strategy's turnover? Would it survive a 50 bps cost assumption?
* Write the one sentence you would put in an investment memo about this backtest.